# RAG Pipeline — Hierarchical RAG

Upgrade over the vanilla RAG baseline. Two-level hierarchy per page: child nodes are Claude-generated one-sentence summaries (sharp retrieval signal); parent nodes are full page text (rich generation context). Retrieval matches on child summaries; `AutoMergingRetriever` promotes to parent for generation.

**Retrieval score: 6/11** vs 4/10 baseline.

## 1. Installation

In [ ]:
!pip install -q -U huggingface_hub
!pip install -Uq llama-index
!pip install -Uq pymupdf llama-index-readers-file
!pip install -Uq llama-index-embeddings-huggingface
!pip install -Uq llama-index-llms-huggingface
!pip install -Uq bitsandbytes
!pip install -Uq nest_asyncio
!pip install -Uq anthropic

## 2. Imports and Initialisation

In [ ]:
import os
import re
import json
import time
import random
import numpy as np
import torch
import nest_asyncio

from google.colab import drive, userdata
from huggingface_hub import login
import anthropic

nest_asyncio.apply()
drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

client_anth = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 3. Data Extraction and Corpus Cleaning

Load the thesis with PyMuPDF. Restrict to pages 39–327 (289 pages): front matter and bibliography excluded. Front matter adds no signal; bibliography pages caused hallucinated summaries (Calabi-Yau geometry, holomorphic 3-forms) in preliminary runs.

In [ ]:
from llama_index.readers.file import PyMuPDFReader

loader = PyMuPDFReader()
docs = loader.load(file_path="/content/drive/MyDrive/SISSA_Thesis_fin_promise.pdf")
print(f"Total pages: {len(docs)}")

content_docs = [doc for doc in docs if
                int(doc.metadata['source']) >= 39 and
                int(doc.metadata['source']) <= 327]

print(f"Clean corpus: {len(content_docs)} pages")

In [ ]:
# Verify boundaries
print("First content page:")
print(content_docs[0].text[:300])
print("\nLast content page:")
print(content_docs[-1].text[:300])

## 4. Summary Generation

Generate one-sentence summaries for each page using `claude-sonnet-4-5`. These become the child nodes — the retrieval signal layer.

**Why Claude:** Llama-3.1-8B hallucinated domain-irrelevant concepts (holomorphic 3-forms, Calabi-Yau geometry). Claude produces physically accurate, terminology-preserving summaries.

**Cost:** ~$5 for 289 pages on claude-sonnet-4-5. Saved to Drive; never needs re-running.

In [ ]:
# Load pre-generated summaries from Drive (skip re-generation if already done)
with open("/content/drive/MyDrive/content_summaries_claude.json", "r") as f:
    summaries = json.load(f)

print(f"Summaries loaded: {len(summaries)}")

# Spot-check key pages
print("\nPage 52 (CS level shift):")
print(summaries['52'])
print("\nPage 313 (Class S footnote):")
print(summaries['313'])
print("\nPage 315 (operator map):")
print(summaries['315'])

In [ ]:
# ── Only run this cell if summaries need to be regenerated ──
# Incremental: skips pages already summarised; safe to interrupt and resume.

summaries = {}

for i, doc in enumerate(content_docs):
    if doc.metadata['source'] in summaries:
        continue

    while True:
        try:
            response = client_anth.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=250,
                messages=[{
                    "role": "user",
                    "content": (
                        "In one sentence, summarise the key physics content of this passage "
                        "from a theoretical physics thesis on 3d mirror symmetry. Be specific "
                        "about equations, dualities, and results mentioned. Do not introduce "
                        "any concepts not explicitly present in the passage.\n\n"
                        f"{doc.text[:3000]}"
                    )
                }]
            )
            summaries[doc.metadata['source']] = response.content[0].text
            if i % 50 == 0:
                print(f"Page {i}/289")
                with open("/content/drive/MyDrive/content_summaries_claude.json", "w") as f:
                    json.dump(summaries, f)
            break

        except Exception as e:
            print(f"Error at page {i}: {e}")
            time.sleep(10)

with open("/content/drive/MyDrive/content_summaries_claude.json", "w") as f:
    json.dump(summaries, f)

print(f"Done. {len(summaries)} summaries saved.")

## 5. Hierarchical Index Construction

For each page, create two linked nodes:
- **Parent node:** full raw page text — used for generation
- **Child node:** one-sentence summary — used for retrieval

Retrieval matches on child vocabulary; `AutoMergingRetriever` walks up to the parent to return full context to the LLM.

In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",
    device="cuda"
)

In [ ]:
nodes = []

for doc in content_docs:
    page = doc.metadata['source']
    summary = summaries.get(page, doc.text[:500])  # fallback: first 500 chars

    # Parent: full page text
    parent_node = TextNode(
        text=doc.text,
        metadata={**doc.metadata, 'node_type': 'parent'},
        id_=f"parent_{page}"
    )

    # Child: summary, linked to parent
    child_node = TextNode(
        text=summary,
        metadata={**doc.metadata, 'node_type': 'child'},
        id_=f"child_{page}",
        relationships={
            NodeRelationship.PARENT: RelatedNodeInfo(node_id=parent_node.node_id)
        }
    )

    nodes.extend([parent_node, child_node])

print(f"Total nodes: {len(nodes)}")  # 289 pages × 2 = 578

In [ ]:
# Store all nodes in docstore (needed for AutoMergingRetriever to walk to parents)
docstore = SimpleDocumentStore()
docstore.add_documents(nodes)

# Build vector index over child nodes only
child_nodes = [n for n in nodes if n.metadata['node_type'] == 'child']
index = VectorStoreIndex(child_nodes, show_progress=True)
print("Index built.")

In [ ]:
# Persist to Drive
index.storage_context.persist(persist_dir="/content/drive/MyDrive/rag_hierarchy_index")
print("Index saved.")

### Reload index (use after runtime restart)

In [ ]:
from llama_index.core import load_index_from_storage

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",
    device="cuda"
)

storage_context = StorageContext.from_defaults(
    persist_dir="/content/drive/MyDrive/rag_hierarchy_index"
)
index = load_index_from_storage(storage_context)

# Rebuild docstore (not persisted — must be reconstructed from docs + summaries)
docstore = SimpleDocumentStore()
docstore.add_documents(nodes)  # requires content_docs and summaries to be in memory
print("Index loaded.")

## 6. Reader LLM

**Note:** Mistral-7B-Instruct 4-bit uses ~13GB of T4 VRAM. Load after the index. If OOM occurs, free memory first with `torch.cuda.empty_cache()`.

In [ ]:
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import PromptTemplate
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

Settings.llm = HuggingFaceLLM(
    model_name="mistralai/Mistral-7B-Instruct-v0.3",
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.3",
    device_map="cuda",
    model_kwargs={"quantization_config": quantization_config}
)

qa_prompt = PromptTemplate(
    "You are a precise physics assistant. Answer ONLY using the context below. "
    "Do not use any prior knowledge. If the answer is not in the context, say 'Not found in context'.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)

query_engine = index.as_query_engine(
    similarity_top_k=5,
    text_qa_template=qa_prompt
)
print("Query engine ready.")

## 7. Retrieval Evaluation

Gold set of 11 domain-expert questions written by the thesis author. Q11 is a smoke test: does the system retrieve the correct 3d definition of mirror symmetry (exchange of order/disorder operators) or hallucinate the Calabi-Yau version?

In [ ]:
questions = [
    "What is mirror symmetry in the context of the theories described in the thesis?",
    "In the fundamental N=2 mirror pair, what are the three operators mapped across the duality, and what is the superpotential on the mirror side?",
    "When a single fermion of positive mass is integrated out in a U(1) gauge theory, what happens to the CS level and what gravitational term is generated?",
    "In the planar mirror of U(N) SQCD with F1+N fundamental and F2+N antifundamental chiral multiplets, what is the UV topological symmetry and what does it enhance to in the IR?",
    "In Figure 1.1, why does applying the S-element of SL(2;Z) to the Hanany-Witten configuration not immediately give a supersymmetric mirror, and what procedure restores supersymmetry?",
    "In Appendix B, after performing all column dualizations, how many singlets are generated in total, and what representation of the flavor symmetry do they reproduce?",
    "In the USp phase diagram (Figure 4.7), what are the qualitatively distinct zones in the (F, |k|) plane, and what distinguishes the planar duals in each zone?",
    "Which three theories are identified as marginally chiral in Figure 3.15, and what is the special feature of their planar mirror duals?",
    "What happens when a single scalar field is integrated out of SU(2) QCD3 with F scalar fields, and does the resulting duality match the general proposal?",
    "What two inequivalent limits can be defined for reducing three dimensional mirrors to two dimensional mirrors?",
    "The thesis contains a footnote stating that the notation S in Class S theories stands for something the author finds disappointing. What does it stand for and why is it mentioned?",
]

ground_truths = [
    "Mirror symmetry in 3d N=4 theories is an infrared duality exchanging Higgs and Coulomb branches, equivalently exchanging topological and flavour symmetries, or order and disorder operators. It is not a geometric equivalence of Calabi-Yau manifolds.",
    "The three operator mappings are: QQ˜ ↔ F[XX˜], M+ ↔ X, M− ↔ X˜. The mirror superpotential is W_mirror = Flip[XX˜].",
    "The CS level shifts by +1/2 and a gravitational Chern-Simons term of magnitude 1 is generated.",
    "UV topological symmetry is U(1)^(F1+F2−1); it enhances to S[U(F1+N) × U(F2+N)] × U(1) in the IR.",
    "The S-element creates a Coulomb force between branes that breaks supersymmetry. Supersymmetry is restored by performing Hanany-Witten brane creation moves, bringing branes inwards.",
    "9 singlets are generated in total, transforming as the fundamental of one SU(3) and the antifundamental of the other SU(3) flavor symmetry.",
    "There are three qualitatively distinct zones in the (F, |k|) plane, each with a qualitatively different dual structure.",
    "The three marginally chiral theories are U(2)_{-1/2} with [2,3], U(2)_0 with [2,2], and U(2)_{-1} with [1,3]. Their planar mirror duals contain a mesonic operator dual to a gauge invariant monopole.",
    "A single column is integrated out, and the resulting duality matches the general proposal.",
    "The two limits are the Higgs limit and the Coulomb limit, producing a GLSM and conformal blocks respectively.",
    "S stands for 'six', referring to the six-dimensional parent SCFT. The author flags this in a footnote as disappointingly mundane.",
]

In [ ]:
# Retrieval evaluation — no LLM needed
base_retriever = index.as_retriever(similarity_top_k=5)
retriever = AutoMergingRetriever(base_retriever, docstore)

for i, q in enumerate(questions):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {q}")
    print('='*60)
    nodes_retrieved = retriever.retrieve(q)
    for j, node in enumerate(nodes_retrieved):
        print(f"  Chunk {j+1} | Page {node.metadata['source']} | Score {node.score:.3f}")
        print(f"  {node.text[:300]}")
        print()

### Hierarchical retrieval results

| Question | Correct page | Vanilla RAG | Hierarchical RAG |
|----------|-------------|-------------|------------------|
| Q1 (mirror symmetry) | 40/301 | — | ✓ |
| Q2 (operator map) | 315 | ✗ | ✗ |
| Q3 (CS level shift) | 52/53 | ✓ | ✓ |
| Q4 (topological symmetry) | 130/134 | ✓ | ✗ |
| Q5 (S-rule) | 77/78 | ✓ | ✓ |
| Q6 (singlet count) | Appendix B | ✗ | ✗ |
| Q7 (USp zones) | 216 | ✓ | ✓ |
| Q8 (marginally chiral) | 164 | ✗ | ✗ |
| Q9 (scalar integrated out) | 285/287 | ✗ | ✓ |
| Q10 (2d limits) | 304/305 | ✗ | ✗ |
| Q11 (Class S footnote) | 313 | ✗ | ✓ |
| **Score** | | **4/10** | **6/11** |

**Persistent failures:**
- Q2: vocabulary collision — multiple pages discuss operator maps; correct page scores below noise floor
- Q6: calculation-buried answer — result implicit in multi-step appendix derivation, never stated as a clean sentence
- Q10: ranking problem — correct page likely at position 6–8; fix with cross-encoder reranking

## 8. Next Steps

- **Reranking:** `BAAI/bge-reranker-large` — expected to fix Q10
- **LlamaParse:** Vision-capable parser recovering compiled TikZ quiver diagrams as figure descriptions
- **RAGAS evaluation:** faithfulness, answer relevancy, context precision, context recall — once generation is stable
- **GraphRAG:** Knowledge graph where nodes are theories/quivers and edges are dualities/RG flows/quiver subtractions. Multi-hop retrieval for questions requiring chained reasoning across the thesis.

 ## 9. Running with reranking

In [ ]:
!pip install -Uq llama-index-postprocessor-flag-embedding-reranker
!pip install -Uq ragas datasets

In [ ]:
from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker
from llama_index.core.retrievers import AutoMergingRetriever

reranker = FlagEmbeddingReranker(
    model="BAAI/bge-reranker-large",
    top_n=5  # rerank top-10 down to top-5
)

base_retriever = index.as_retriever(similarity_top_k=10)
retriever = AutoMergingRetriever(base_retriever, docstore)

In [ ]:
from llama_index.core import StorageContext
from llama_index.core.retrievers import AutoMergingRetriever

storage_context_with_docs = StorageContext.from_defaults(docstore=docstore)

base_retriever = index.as_retriever(similarity_top_k=10)
retriever = AutoMergingRetriever(base_retriever, storage_context_with_docs)

In [ ]:
questions = [
    "What is mirror symmetry in the context of the theories described in the thesis?",
    "In the fundamental N=2 mirror pair, what are the three operators mapped across the duality, and what is the superpotential on the mirror side?",
    "When a single fermion of positive mass is integrated out in a U(1) gauge theory, what happens to the CS level and what gravitational term is generated?",
    "In the planar mirror of U(N) SQCD with F1+N fundamental and F2+N antifundamental chiral multiplets, what is the UV topological symmetry and what does it enhance to in the IR?",
    "In Figure 1.1, why does applying the S-element of SL(2;Z) to the Hanany-Witten configuration not immediately give a supersymmetric mirror, and what procedure restores supersymmetry?",
    "In Appendix B, after performing all column dualizations, how many singlets are generated in total, and what representation of the flavor symmetry do they reproduce?",
    "In the USp phase diagram (Figure 4.7), what are the qualitatively distinct zones in the (F, |k|) plane, and what distinguishes the planar duals in each zone?",
    "Which three theories are identified as marginally chiral in Figure 3.15, and what is the special feature of their planar mirror duals?",
    "What happens when a single scalar field is integrated out of SU(2) QCD3 with F scalar fields, and does the resulting duality match the general proposal?",
    "What two inequivalent limits can be defined for reducing three dimensional mirrors to two dimensional mirrors?",
    "The thesis contains a footnote stating that the notation S in Class S theories stands for something the author finds disappointing. What does it stand for and why is it mentioned?",
]

contexts = []
retrieved_pages = []

for i, q in enumerate(questions):
    nodes_retrieved = retriever.retrieve(q)
    reranked = reranker.postprocess_nodes(nodes_retrieved, query_str=q)

    context_texts = [n.text for n in reranked]
    pages = [n.metadata['source'] for n in reranked]

    contexts.append(context_texts)
    retrieved_pages.append(pages)

    print(f"\n{'='*60}")
    print(f"Q{i+1}: {q}")
    print('='*60)
    for j, node in enumerate(reranked):
        print(f"  Chunk {j+1} | Page {node.metadata['source']} | Score {node.score:.3f}")
        print(f"  {node.text[:300]}")
        print()